##### Features and targets

Here, we prepare the model features for **every trading day**. A row dated *t* uses only data dated *t* or earlier.

**Label and horizon.** The prediction target is `fwd_spy_xs`, which is already in the panel, SPY minus BIL total return
over close(t+1) to close(t+2). So the horizon is **H = 1 trading day**:

A model refitted on day *T* may only train on rows *t* ≤ *T* − 2, because later labels are not yet realised.
Additionally, we provide the 5- and 10-day targets in `src/features/targets.py` for robustness checks (training rows
*t* ≤ *T* − 1 − *h*)

**Design and test periods.** Every feature-related design choice in this notebook looks only at the design period
2014–2018 (`config.DESIGN_END`).

**Output** `data/processed/features.parquet` (with a `.csv` version) and `data/processed/feature_sets.json`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config
from src.data.build import load_panel
from src.features import events

panel = load_panel()
s = panel["in_sample"] # samples without the warmup period
d = s & (panel.index <= config.DESIGN_END)   # design period, 2014-2018
print(panel.shape, panel.index.min().date(), "->", panel.index.max().date(), f"({s.sum()} trading days in sample)")

(3530, 34) 2012-01-03 -> 2026-01-15 (3018 trading days in sample)


In [2]:
cols = ["spy_adj_close", "spy_tr_cc", "vix_close", "vix3m_close", "vix_ratio", "fwd_spy_xs"]
display(panel.loc[s, cols].describe().T.round(4))
print("NaN in sample:", int(panel.loc[s, cols].isna().sum().sum()))
print("fwd_spy_xs[t] == spy_xs_cc[t+2]:",
      np.allclose(panel["fwd_spy_xs"], panel["spy_xs_cc"].shift(-2), equal_nan=True))

,count,mean,std,min,25%,50%,75%,max
spy_adj_close,3018.0,322.8707,143.6480,141.7395,195.5039,272.9063,416.0080,690.3800
spy_tr_cc,3018.0,0.0006,0.0109,-0.1094,-0.0036,0.0007,0.0058,0.1050
vix_close,3018.0,17.9843,6.9631,9.1400,13.3700,16.1500,20.7100,82.6900
vix3m_close,3018.0,19.9033,6.1617,11.8500,15.6000,18.2750,22.3725,72.9800
vix_ratio,3018.0,0.8911,0.0778,0.7359,0.8368,0.8798,0.9351,1.3437
fwd_spy_xs,3018.0,0.0005,0.0109,-0.1095,-0.0037,0.0007,0.0057,0.1050


NaN in sample: 0
fwd_spy_xs[t] == spy_xs_cc[t+2]: True


#### 2. Feature definitions

We propose the following candidate features with there definition and the reasoning. 

- **Decline state**: how far and how fast SPY has fallen.
  - `dec_dd` = close / 20-day high − 1. *Lookback 20 days.*
    - This describes the depth of the current pull-back. Short-term losses in the index tend to partly reverse, and this is part of the decline trigger.
  - `dec_mom` = 5-day return / (20-day vol × √5). *Lookback 5 days, scaled by 20-day vol.*
    - It represents the speed of the sell-off as a z-score, so identical levels of decline are adjusted differently according to volatility.
  - `dec_vol` = standard deviation of daily SPY total returns. *Lookback 20 days.*
    - Recent realised risk. High recent volatility tends to come with lower risk-adjusted returns ahead
  - `dec_cond` = 1 if `dec_dd` ≤ −2% and `dec_mom` ≤ −1, else 0. *Lookback 20 days.*
    - It marks the decline regime we study, and helps switch the interaction term on.
- **Trend**: whether the longer-run uptrend is intact.
  - `ma200_gap` = close / 200-day moving average − 1. *Lookback 200 days.*
    - It captures a long-term dynamic and denotes different dynamics according to market conditions.
- **VIX level**: how much fear is priced in, and in which direction it is moving.
  - `vix_log` = log VIX. *Same-day close.*
    - The level of priced fear. Logs, because VIX is strongly right-skewed.
  - `vix_z` = (VIX − 252-day mean) / 252-day std. *Lookback 252 days.*
    - Fear relative to the past year, which removes slow shifts in the typical VIX level.
  - `vix_chg5` = log change of VIX. *Lookback 5 days, matching `dec_mom`.*
    - Whether fear is still rising or already fading.
- **Term structure**: whether the market sees the stress as short-lived or lasting.
  - `ts_ratio` = VIX / VIX3M. *Same-day close (30-day vs 3-month implied vol).*
    - Above 1 (backwardation) means near-term fear exceeds 3-month fear: acute stress that the market expects
      to fade, typical near a capitulation low. This is our project's main signal.
  - `ts_backw` = 1 if `ts_ratio` > 1, else 0. *Same day.*
    - A discrete regime flag, insensitive to small moves of the ratio around its typical 0.9.
  - `ts_chg5` = 5-day change of `ts_ratio`. *Lookback 5 days.*
- **Implied vs realised volatility**: whether options price more fear than the market has shown.
  - `vrp` = VIX/100 − annualised 20-day realised vol. *Lookback 20 days.*
    - Positive when investors pay up for protection; it turns negative in crashes, when realised volatility
      jumps faster than implied.
- **Interactions**: they tell whether the term-structure signal behave differently during declines.
  - `x_cond_ts` = `dec_cond` × (`ts_ratio` − 1).
    - In $y = b_1\,TS + b_2\,D + b_3\,(D \times TS)$ the term-structure slope is $b_1$ on normal days and
      $b_1 + b_3$ on decline days, which is the research question.
  - `x_dd_ts` = `dec_dd` × (`ts_ratio` − 1). 
  - `x_mom_backw` = `dec_mom` × `ts_backw`.
    - Lets the speed of the sell-off matter differently in backwardation: a sharp drop while near-term fear
      exceeds 3-month fear is the clearest picture of capitulation.
  - `x_mom_ts` = `dec_mom` × (`ts_ratio` − 1).
    - Continuous version of `x_mom_backw`: uses the ratio on every day instead of only in backwardation.
  - `x_cond_vrp` = `dec_cond` × `vrp`. 
  - `x_cond_vixchg` = `dec_cond` × `vix_chg5`. 

The term-structure ratio enters the interactions as `ts_ratio − 1` (distance from backwardation). Since the raw
ratio sits around 0.9, `dec_cond × ts_ratio` would be almost the same column as `dec_cond`; centring on a sample
mean would leak future data, while the constant 1 is fixed in advance.

In [3]:
# The functions live in src/features/feature_build.py so scripts and tests can import them.
# Shown here for reading; edit the module, not this cell.
import inspect
from src.features import feature_build as fb

for fn in (fb.decline_features, fb.vix_features, fb.interactions, fb.build_candidates):
    print(inspect.getsource(fn))

feat = fb.build_candidates(panel)          # all 18 candidates, before pruning
feat.loc[s].describe().T.round(4)

def decline_features(panel: pd.DataFrame) -> pd.DataFrame:
    f = events.decline_signals(panel)                          # dec_dd, dec_mom, dec_vol
    f["dec_cond"] = ((f["dec_dd"] <= -events.S) & (f["dec_mom"] <= -events.P)).astype(float)
    a = panel["spy_adj_close"]
    f["ma200_gap"] = a / a.rolling(200).mean() - 1
    return f

def vix_features(panel: pd.DataFrame) -> pd.DataFrame:
    vix, ratio = panel["vix_close"], panel["vix_ratio"]
    return pd.DataFrame({
        "vix_log": np.log(vix),
        "vix_z": (vix - vix.rolling(252).mean()) / vix.rolling(252).std(),
        "vix_chg5": np.log(vix / vix.shift(5)),
        "ts_ratio": ratio,
        "ts_backw": (ratio > 1).astype(float),
        "ts_chg5": ratio - ratio.shift(5),
    })

def interactions(f: pd.DataFrame) -> pd.DataFrame:
    ts = f["ts_ratio"] - 1          # distance from backwardation; a fixed centre, so no leakage
    return pd.DataFrame({
        "x_cond_ts": f["dec_cond"] * ts,
        "x_dd_ts": f["dec_dd"]

,count,mean,std,min,25%,50%,75%,max
dec_dd,3018.0,-0.0181,0.0289,-0.2911,-0.0235,-0.0070,-0.0003,0.0000
dec_mom,3018.0,0.2090,0.9743,-3.7506,-0.3819,0.2752,0.8738,3.4825
dec_vol,3018.0,0.0092,0.0061,0.0020,0.0056,0.0076,0.0110,0.0587
dec_cond,3018.0,0.0958,0.2943,0.0000,0.0000,0.0000,0.0000,1.0000
ma200_gap,3018.0,0.0525,0.0606,-0.2566,0.0260,0.0633,0.0909,0.1786
vix_log,3018.0,2.8329,0.3207,2.2127,2.5930,2.7819,3.0306,4.4151
vix_z,3018.0,-0.0346,1.2741,-2.0558,-0.8760,-0.3101,0.4635,11.8278
vix_chg5,3018.0,0.0001,0.1636,-0.6287,-0.0903,-0.0121,0.0765,1.1407
ts_ratio,3018.0,0.8911,0.0778,0.7359,0.8368,0.8798,0.9351,1.3437
ts_backw,3018.0,0.0759,0.2648,0.0000,0.0000,0.0000,0.0000,1.0000


#### 3. Checks

**Coverage.** Every in-sample day must have every feature (the 2012–13 warm-up covers the 252-day VIX window).

**Leakage.** Rebuild the features on the panel cut at several dates and confirm the values up to the cut are
identical.

In [4]:
nan_in_sample = feat.loc[s].isna().sum()
assert nan_in_sample.sum() == 0, nan_in_sample[nan_in_sample > 0]
assert not any(c.startswith("fwd_") for c in feat.columns)

for frac in (0.25, 0.5, 0.75, 0.95):
    cut = panel.index[int(len(panel) * frac)]
    pd.testing.assert_frame_equal(feat.loc[:cut], fb.build_candidates(panel.loc[:cut]), check_exact=True)
print("coverage and truncation checks passed")

coverage and truncation checks passed


#### 4. Feature selection

`feature_build.select_features` applies the following rule.

1. **Interaction that is a rescaled part.** Drop an interaction whose |corr| with one of its own parts exceeds
   0.8 for its insignificant information gain. Then, since VIX almost always rises on decline days, `dec_cond × vix_chg5` is
   nearly a rescaled `dec_cond`, and the same holds for `dec_cond × vrp`.
2. **Near-duplicates.** Of two candidates with |corr| above 0.9, we drop the one listed later.
3. **Abundance of term-structure interaction.** Since these interactions are informative only on the few days
   when both parts are active, we keep one only if it has at least 15 separate episodes of material variation in the
   design period. Which is roughly three times a year. An episode counts if there are two consecutive active interaction
   term within 10 days.

The correlation table below is computed on the design period. For pairs having correlation above 0.7, we still kept them as 
each of it answers a different question (for example `vix_z` and `ts_ratio` represents the fear level and the term-structure shape respectively).

In [5]:
c = feat.loc[d].corr()
pairs = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1)).stack()
pairs[pairs.abs() > 0.7].sort_values(key=abs, ascending=False).round(3).to_frame("corr (2014-2018)")

corr (2014-2018)
vix_chg5    ts_chg5                   0.950
x_cond_vrp  x_cond_vixchg             0.897
vix_z       ts_ratio                  0.878
vix_log     vix_z                     0.860
dec_cond    x_cond_vrp                0.852
vix_log     ts_ratio                  0.852
dec_cond    x_cond_vixchg             0.832
dec_dd      vix_log                  -0.818
            ts_ratio                 -0.800
x_mom_backw x_cond_vixchg            -0.796
ma200_gap   vix_log                  -0.790
dec_vol     vix_log                   0.789
dec_dd      vix_z                    -0.789
dec_mom     vix_chg5                 -0.773
x_cond_ts   x_dd_ts                  -0.763
dec_dd      ma200_gap                 0.711

In [6]:
# rules always run on the full candidate set, so this cell can be re-run safely
dropped, log = fb.select_features(fb.build_candidates(panel).loc[d])
assert dropped == fb.DROPPED, dropped
display(log)

feat = feat.drop(columns=fb.DROPPED, errors="ignore")
print(f"In total {feat.shape[1]} features kept")

,feature,decision,reason
0,x_cond_vrp,drop,rule 1: |corr| 0.85 with its own part
1,x_cond_vixchg,drop,rule 1: |corr| 0.83 with its own part
2,ts_chg5,drop,rule 2: |corr| 0.95 with vix_chg5
3,x_cond_ts,keep,rule 3: 25 material episodes (min 15)
4,x_dd_ts,drop,rule 3: 9 material episodes (min 15)
5,x_mom_backw,keep,rule 3: 17 material episodes (min 15)
6,x_mom_ts,keep,rule 3: 26 material episodes (min 15)


In total 14 features kept


#### 5. Feature sets

Then, we sort the kept features into three nested sets.

- `base` (5): price-based decline state and trend only
- `vix` (11): `base` plus VIX level, term structure and the implied-vs-realised gap
- `full` (14): `vix` plus the three decline × term-structure interactions, `x_cond_ts`, `x_mom_backw` and `x_mom_ts`

In the later performance evaluation, comparing the three sets tests whether the VIX information, and then the
interaction, adds predictive power. Each step adds few parameters, so the comparison stays fair.

In [7]:
FEATURE_SETS = fb.FEATURE_SETS             # set in feature_build.py
assert set(FEATURE_SETS["full"]) == set(feat.columns)
pd.testing.assert_frame_equal(feat[FEATURE_SETS["full"]], fb.build_features(panel))
{k: len(v) for k, v in FEATURE_SETS.items()}

{'base': 5, 'vix': 11, 'full': 14}

**Decline days.** How many in-sample days have the trigger on, and how often it coincides with backwardation.

In [8]:
print(pd.crosstab(feat.loc[s, "dec_cond"], feat.loc[s, "ts_backw"],
                  rownames=["dec_cond"], colnames=["ts_backw"], margins=True))

ts_backw   0.0  1.0   All
dec_cond                 
0.0       2614  115  2729
1.0        175  114   289
All       2789  229  3018


#### 6. Cleaning and scaling

**Missing data.** The panel has no gaps in 2014–2025 (VIX and VIX3M are by default forward-filled for at most one day
upstream, yet that never happened in the sample). The longest lookback is 252 days, covered by the 2012–13
warm-up, so every in-sample day has every feature.

**Outliers.** The saved features are raw and nothing is clipped or deleted, because the extreme days (for example
March 2020) are exactly the declines we study. Clipping and standardising are done inside each training window by
`fb.PastScaler`:

- continuous features are clipped at the 1% and 99% quantiles of the training rows, then standardised with the
  training mean and standard deviation;
- the same bounds, means and standard deviations are applied unchanged to the later rows being predicted;
- binary flags (`dec_cond`, `ts_backw`) and the interaction are only standardised while they are mostly 0.

The table below shows the most extreme raw values in the sample, then the scaler fitted on the design period and
applied to the test period.

In [9]:
ext = pd.DataFrame({"min": feat.loc[s].min(), "date_min": feat.loc[s].idxmin().dt.date,
                    "max": feat.loc[s].max(), "date_max": feat.loc[s].idxmax().dt.date})
display(ext.loc[["vix_z", "vix_chg5", "vrp", "dec_vol", "dec_dd"]].round(3))

sc = fb.PastScaler().fit(feat.loc[d])                     # fit on the design period only
z_test = sc.transform(feat.loc[s & ~d])                   # apply to 2019-2025
z_test.describe().T[["mean", "std", "min", "max"]].round(2)

,min,date_min,max,date_max
vix_z,-2.056,2023-06-22,11.828,2018-02-05
vix_chg5,-0.629,2024-12-26,1.141,2015-08-24
vrp,-0.460,2020-04-06,0.229,2021-01-27
dec_vol,0.002,2017-08-09,0.059,2020-03-27
dec_dd,-0.291,2020-03-16,0.000,2014-02-24


,mean,std,min,max
dec_dd,-0.24,1.28,-4.09,0.71
dec_mom,0.04,1.02,-2.59,2.31
dec_vol,0.66,1.17,-1.20,3.24
dec_cond,0.03,1.04,-0.32,3.17
ma200_gap,0.27,1.59,-3.31,1.91
vix_log,1.06,1.03,-0.87,2.73
vix_z,-0.08,1.01,-1.37,3.69
vix_chg5,-0.03,0.95,-2.42,3.22
ts_ratio,0.20,0.89,-1.61,3.43
ts_backw,-0.06,0.90,-0.31,3.26


#### 7. Save

In [10]:
# Same as running: python -m src.features.feature_build
fb.save(fb.build_features(panel))

wrote 14 features x 3530 rows to data/processed


To using the features, please do the following

```python
import json, pandas as pd
from src import config
from src.data.build import load_panel

panel = load_panel()
feat = pd.read_parquet(config.PROCESSED_DIR / "features.parquet")
sets = json.loads((config.PROCESSED_DIR / "feature_sets.json").read_text())
X, y = feat[sets["full"]], panel["fwd_spy_xs"]

# inside each refit on day T: ONLY train on rows t <= T - 2, scale with training statistics only
from src.features.feature_build import PastScaler
sc = PastScaler().fit(X_train)
X_train_z, X_new_z = sc.transform(X_train), sc.transform(X_new)
```